# HiStrA Python vs C# Verification & Bridge Scenario Analysis

This notebook demonstrates the in-process Python HiStrA solver across **benchmark models** (`benchmark_1`, `benchmark_2`) and **full-scale 3D masonry bridge models** (`random_001` through `random_006`), comparing Python execution against authoritative C# `.Results` databases.

### Verification & Engineering Analysis Workflows:
1. **Equilibrium & Capacity Curves**: Vertical Reaction / Balancing Reaction vs Pier Top Settlement ($R_z$ vs $U_z$).
2. **Scour Progression Analysis**: Upstream Scour Depth Ratio ($0\% 	o 20\% 	o 40\%$) vs Pier Settlement ($U_z$) and Transverse Drift ($U_y$).
3. **Pier Top Kinematics**: Displacements ($U_x, U_y, U_z$) at the top of the pier where superstructure loads are transferred.
4. **Pier Rotation & Tilt**: Transverse rotation ($\theta_x = \Delta U_z / B_{pier}$) and longitudinal tilt ($\theta_y = \Delta U_x / H_{pier}$) in milliradians.
5. **Multi-Model Batch Parity Benchmark**: Cross-model comparison across all 6 random models.


In [ ]:
from __future__ import annotations

import copy
import json
import math
from pathlib import Path
import sqlite3
import time
from typing import Any, Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np

from histra.io.hr_loader import load_model
from histra.solver import AnalysisSession
from histra.solver.model_manager import ModelManager
from histra.solver.output_projection import compute_model_point_displacements

# Matplotlib formatting and visualization style
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 10
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["legend.fontsize"] = 9
plt.rcParams["grid.alpha"] = 0.5
plt.rcParams["grid.linestyle"] = "--"
plt.rcParams["figure.autolayout"] = True

print("HiStrA Python environment loaded successfully.")


In [ ]:
# ==============================================================================
# Geometry and Result Synchronization Helpers
# ==============================================================================

SOIL_MATERIAL_KEY = 146
SOIL_REMOVED_MATERIAL_KEY = 147
PIER_CENTRES_X = (502.0, 846.0)
PIER_X = 502.0
PIER_LENGTH = 139.8
PIER_Y = 0.0
PIER_WIDTH = 342.4
PIER_HEIGHT = 225.0  # Height from foundation base (z=-225) to top (z=0)


def _centre(interface) -> tuple[float, float, float]:
    vertices = interface.vint3d
    return tuple(sum(getattr(v, axis) for v in vertices) / 4.0 for axis in ("x", "y", "z"))


def pier_foundation_interface_keys(
    model,
    pier_centres_x: tuple[float, ...] = PIER_CENTRES_X,
    pier_length: float = PIER_LENGTH,
) -> list[int]:
    half_len = pier_length / 2.0
    return sorted(
        int(intf.key)
        for intf in model.collections.interfaces.values()
        if "Restraint" in (intf.parent_type_element1, intf.parent_type_element2)
        and any(abs(_centre(intf)[0] - px) <= half_len + 1.0e-4 for px in pier_centres_x)
    )


def upstream_interface_keys(
    model,
    delta: float,
    pier_x: float = PIER_X,
    pier_length: float = PIER_LENGTH,
    pier_y: float = PIER_Y,
    pier_width: float = PIER_WIDTH,
) -> list[int]:
    left = pier_x - pier_length / 2.0
    right = pier_x + pier_length / 2.0
    upstream = pier_y - pier_width / 2.0
    limit = upstream + pier_width * float(delta)
    selected: list[int] = []
    for intf in model.collections.interfaces.values():
        if "Restraint" not in (intf.parent_type_element1, intf.parent_type_element2):
            continue
        x, y, _ = _centre(intf)
        if left - 1e-4 <= x <= right + 1e-4 and upstream - 1e-4 <= y <= limit + 1e-4:
            selected.append(int(intf.key))
    return sorted(selected)


def build_aligned_step_history(
    model,
    executions: list,
    csharp_results_path: Path | str,
    preferred_mp_key: int | None = None,
) -> dict:
    """Synchronize Python and C# steps strictly by (AnalysisKey, Step) pairs."""
    csharp_results_path = Path(csharp_results_path)
    cs_reactions = {}
    cs_mps = {}

    if csharp_results_path.exists():
        with sqlite3.connect(csharp_results_path) as db:
            tables = [r[0] for r in db.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()]
            if "ReactionSumStates" in tables:
                r_rows = db.execute(
                    "SELECT AnalysisKey, Step, R1, R2, R3 FROM ReactionSumStates ORDER BY AnalysisKey, Step"
                ).fetchall()
                cs_reactions = {(r[0], r[1]): (r[2], r[3], r[4]) for r in r_rows}
            if "DisplModelPoints" in tables:
                mp_rows = db.execute(
                    "SELECT AnalysisKey, IdElement, ParentKey, Step, Ux, Uy, Uz FROM DisplModelPoints ORDER BY AnalysisKey, Step, IdElement"
                ).fetchall()
                for r in mp_rows:
                    key = (r[0], r[3])  # (AnalysisKey, Step)
                    if key not in cs_mps:
                        cs_mps[key] = {}
                    cs_mps[key][r[2]] = (r[4], r[5], r[6])

    # Determine default ModelPoint
    available_mp_keys = sorted(model.collections.model_points.keys())
    mp_key = preferred_mp_key if preferred_mp_key in available_mp_keys else (available_mp_keys[0] if available_mp_keys else None)

    aligned_steps = []
    
    for execution in executions:
        ak = getattr(execution, "analysis_key", None)
        if ak is None:
            for a in model.collections.analyses.values():
                if a.name == execution.analysis_name:
                    ak = int(a.key)
                    break
            if ak is None:
                ak = 1

        aname = execution.analysis_name

        # Initial state (Step 0)
        if (ak, 0) in cs_reactions and execution.committed_steps:
            cs_r = cs_reactions[(ak, 0)]
            cs_mp_dict = cs_mps.get((ak, 0), {})
            py_init_u = execution.initial_step.u if hasattr(execution, "initial_step") and execution.initial_step is not None else np.zeros(model.gdl)
            py_mp_dict = {
                m.parent_key: (m.ux, m.uy, m.uz)
                for m in compute_model_point_displacements(model, py_init_u, step=0)
            }
            aligned_steps.append({
                "label": f"{aname}:0",
                "analysis_key": ak,
                "analysis_name": aname,
                "step": 0,
                "py_r3": execution.initial_step.reaction_z if hasattr(execution, "initial_step") and execution.initial_step is not None and execution.initial_step.reaction_z is not None else 0.0,
                "cs_r3": cs_r[2],
                "py_mps": py_mp_dict,
                "cs_mps": cs_mp_dict,
            })

        for step in execution.committed_steps:
            s_num = step.step
            cs_r = cs_reactions.get((ak, s_num), (0.0, 0.0, 0.0))
            cs_mp_dict = cs_mps.get((ak, s_num), {})
            py_mp_dict = {
                m.parent_key: (m.ux, m.uy, m.uz)
                for m in compute_model_point_displacements(model, step.u, step=s_num)
            }
            aligned_steps.append({
                "label": f"{aname}:{s_num}",
                "analysis_key": ak,
                "analysis_name": aname,
                "step": s_num,
                "py_r3": step.reaction_z,
                "cs_r3": cs_r[2],
                "py_mps": py_mp_dict,
                "cs_mps": cs_mp_dict,
            })

    return {
        "steps": aligned_steps,
        "selected_mp_key": mp_key,
        "has_csharp": bool(cs_reactions),
    }


def extract_scour_progression(
    model,
    executions: list,
    csharp_results_path: Path | str,
    scour_stages: list[tuple[float, str]],  # [(0.0, "Vert"), (0.2, "scour_1"), (0.4, "scour_2"), ...]
) -> dict:
    """Extract final converged equilibrium states across sequential scour depth ratios."""
    csharp_results_path = Path(csharp_results_path)
    cs_mps_by_ak = {}

    if csharp_results_path.exists():
        with sqlite3.connect(csharp_results_path) as db:
            tables = [r[0] for r in db.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()]
            if "DisplModelPoints" in tables:
                rows = db.execute(
                    "SELECT AnalysisKey, ParentKey, Ux, Uy, Uz FROM DisplModelPoints WHERE Step=(SELECT MAX(Step) FROM DisplModelPoints AS d2 WHERE d2.AnalysisKey=DisplModelPoints.AnalysisKey)"
                ).fetchall()
                for r in rows:
                    if r[0] not in cs_mps_by_ak:
                        cs_mps_by_ak[r[0]] = {}
                    cs_mps_by_ak[r[0]][r[1]] = (r[2], r[3], r[4])

    results = []
    for delta, aname in scour_stages:
        exec_match = [e for e in executions if e.analysis_name == aname]
        if not exec_match or not exec_match[0].committed_steps:
            continue
        
        execution = exec_match[0]
        ak = getattr(execution, "analysis_key", None)
        if ak is None:
            for a in model.collections.analyses.values():
                if a.name == aname:
                    ak = int(a.key)
                    break
            if ak is None:
                ak = 1

        py_u = execution.committed_steps[-1].u
        py_mps = {
            m.parent_key: (m.ux, m.uy, m.uz)
            for m in compute_model_point_displacements(model, py_u, step=execution.committed_steps[-1].step)
        }
        cs_mps = cs_mps_by_ak.get(ak, {})

        entry = {"delta": delta, "analysis_name": aname, "analysis_key": ak}
        
        # Pier 1 Kinematics (MP 8 top, MP 9 upstream, MP 10 downstream, MP 14 base)
        if 8 in py_mps:
            entry["p1_top_ux_py"] = py_mps[8][0] * 1000.0
            entry["p1_top_uy_py"] = py_mps[8][1] * 1000.0
            entry["p1_top_uz_py"] = py_mps[8][2] * 1000.0
        if 8 in cs_mps:
            entry["p1_top_ux_cs"] = cs_mps[8][0] * 1000.0
            entry["p1_top_uy_cs"] = cs_mps[8][1] * 1000.0
            entry["p1_top_uz_cs"] = cs_mps[8][2] * 1000.0

        # Pier 1 Tilts
        if 9 in py_mps and 10 in py_mps:
            entry["p1_theta_x_py"] = (py_mps[10][2] - py_mps[9][2]) / 252.0 * 1000.0  # mrad
        if 9 in cs_mps and 10 in cs_mps:
            entry["p1_theta_x_cs"] = (cs_mps[10][2] - cs_mps[9][2]) / 252.0 * 1000.0

        if 8 in py_mps and 14 in py_mps:
            entry["p1_theta_y_py"] = (py_mps[8][0] - py_mps[14][0]) / 225.0 * 1000.0
        if 8 in cs_mps and 14 in cs_mps:
            entry["p1_theta_y_cs"] = (cs_mps[8][0] - cs_mps[14][0]) / 225.0 * 1000.0

        results.append(entry)

    return {"stages": results}


In [ ]:
# ==============================================================================
# Robust Plotting Routines (Synchronized Step by Step)
# ==============================================================================

def plot_displacement_vs_load(
    aligned_data: dict,
    title: str = "Displacement vs Load (Equilibrium Curves)",
):
    steps = aligned_data["steps"]
    mp_key = aligned_data.get("selected_mp_key")
    has_cs = aligned_data.get("has_csharp", False)

    if not steps:
        print("No step data available to plot.")
        return

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    labels = [s["label"] for s in steps]
    x_indices = np.arange(len(steps))
    py_r3 = [abs(s["py_r3"]) for s in steps]
    cs_r3 = [abs(s["cs_r3"]) for s in steps]

    # Subplot 1: Reaction vs Step Sequence
    ax1.plot(x_indices, py_r3, "o-", color="#1f77b4", label="Python Solver", lw=2, ms=4)
    if has_cs:
        ax1.plot(x_indices, cs_r3, "s--", color="#d62728", label="C# Baseline", lw=1.5, ms=4, alpha=0.8)
    ax1.set_title("Total Vertical Reaction (|R3|) vs Step Sequence")
    ax1.set_xlabel("Analysis Step Sequence")
    ax1.set_ylabel("Vertical Reaction |R3| (kN)")
    
    # Adjust ticks density
    stride = max(1, len(steps) // 15)
    ax1.set_xticks(x_indices[::stride])
    ax1.set_xticklabels(labels[::stride], rotation=45, ha="right", fontsize=8)
    ax1.legend()
    ax1.grid(True)

    # Subplot 2: Equilibrium Capacity Curve (Uz vs R3)
    if mp_key is not None:
        py_uz = [abs(s["py_mps"].get(mp_key, (0, 0, 0))[2]) * 1000.0 for s in steps]
        cs_uz = [abs(s["cs_mps"].get(mp_key, (0, 0, 0))[2]) * 1000.0 for s in steps]

        ax2.plot(py_uz, py_r3, "o-", color="#1f77b4", label=f"Python (MP {mp_key})", lw=2, ms=4)
        if has_cs and any(cs_uz):
            ax2.plot(cs_uz, cs_r3, "s--", color="#d62728", label=f"C# (MP {mp_key})", lw=1.5, ms=4, alpha=0.8)
        ax2.set_title(f"Capacity Curve: Settlement |Uz| vs |R3| (MP {mp_key})")
        ax2.set_xlabel(f"ModelPoint {mp_key} Settlement |Uz| (mm)")
        ax2.set_ylabel("Vertical Load |R3| (kN)")
        ax2.legend()
        ax2.grid(True)

    fig.suptitle(title, fontsize=13, fontweight="bold")
    plt.show()


def plot_scour_vs_displacement(
    scour_data: dict,
    title: str = "Scour Progression vs Pier Displacement",
):
    stages = scour_data.get("stages", [])
    if not stages:
        print("No scour progression data available.")
        return

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    deltas = [s["delta"] for s in stages]
    pct_labels = [f"{int(d * 100)}%" for d in deltas]

    py_uz = [s.get("p1_top_uz_py", 0.0) for s in stages]
    cs_uz = [s.get("p1_top_uz_cs", 0.0) for s in stages]

    py_uy = [s.get("p1_top_uy_py", 0.0) for s in stages]
    cs_uy = [s.get("p1_top_uy_cs", 0.0) for s in stages]

    # 1. Vertical Settlement Uz
    ax1.plot(deltas, py_uz, "o-", color="#1f77b4", label="Python Pier 1 Uz", lw=2, ms=6)
    if any(cs_uz):
        ax1.plot(deltas, cs_uz, "s--", color="#d62728", label="C# Pier 1 Uz", lw=1.5, ms=6, alpha=0.8)
    ax1.set_title("Pier Top Settlement (Uz) vs Scour Depth")
    ax1.set_xlabel("Upstream Scour Depth Ratio")
    ax1.set_ylabel("Pier Top Settlement Uz (mm)")
    ax1.set_xticks(deltas)
    ax1.set_xticklabels(pct_labels)
    ax1.legend()
    ax1.grid(True)

    # 2. Transverse Drift Uy
    ax2.plot(deltas, py_uy, "o-", color="#2ca02c", label="Python Pier 1 Uy", lw=2, ms=6)
    if any(cs_uy):
        ax2.plot(deltas, cs_uy, "s--", color="#ff7f0e", label="C# Pier 1 Uy", lw=1.5, ms=6, alpha=0.8)
    ax2.set_title("Transverse Lateral Drift (Uy) vs Scour Depth")
    ax2.set_xlabel("Upstream Scour Depth Ratio")
    ax2.set_ylabel("Pier Top Transverse Drift Uy (mm)")
    ax2.set_xticks(deltas)
    ax2.set_xticklabels(pct_labels)
    ax2.legend()
    ax2.grid(True)

    fig.suptitle(title, fontsize=13, fontweight="bold")
    plt.show()


def plot_pier_top_kinematics(
    aligned_data: dict,
    title: str = "Pier Top Kinematics (Ux, Uy, Uz)",
    pier_mp: int = 8,
):
    steps = aligned_data["steps"]
    has_cs = aligned_data.get("has_csharp", False)

    if not steps:
        return

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    x_indices = np.arange(len(steps))
    labels = [s["label"] for s in steps]
    stride = max(1, len(steps) // 12)

    components = [
        (0, "Longitudinal Displacement Ux (mm)", axes[0], "#1f77b4", "#d62728"),
        (1, "Transverse Displacement Uy (mm)", axes[1], "#2ca02c", "#ff7f0e"),
        (2, "Vertical Displacement Uz (mm)", axes[2], "#9467bd", "#8c564b"),
    ]

    for axis_idx, ax_title, ax, c_py, c_cs in components:
        py_vals = [s["py_mps"].get(pier_mp, (0, 0, 0))[axis_idx] * 1000.0 for s in steps]
        cs_vals = [s["cs_mps"].get(pier_mp, (0, 0, 0))[axis_idx] * 1000.0 for s in steps]

        ax.plot(x_indices, py_vals, "o-", color=c_py, label=f"Python (MP {pier_mp})", lw=2, ms=3)
        if has_cs and any(cs_vals):
            ax.plot(x_indices, cs_vals, "s--", color=c_cs, label=f"C# (MP {pier_mp})", lw=1.5, ms=3, alpha=0.8)
        ax.set_title(ax_title)
        ax.set_xlabel("Step Sequence")
        ax.set_ylabel("Displacement (mm)")
        ax.set_xticks(x_indices[::stride])
        ax.set_xticklabels(labels[::stride], rotation=45, ha="right", fontsize=8)
        ax.legend()
        ax.grid(True)

    fig.suptitle(title, fontsize=13, fontweight="bold")
    plt.show()


def plot_pier_rotation(
    scour_data: dict,
    title: str = "Pier Tilt & Rotation Angles",
):
    stages = scour_data.get("stages", [])
    if not stages:
        return

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    deltas = [s["delta"] for s in stages]
    pct_labels = [f"{int(d * 100)}%" for d in deltas]

    py_tx = [s.get("p1_theta_x_py", 0.0) for s in stages]
    cs_tx = [s.get("p1_theta_x_cs", 0.0) for s in stages]

    py_ty = [s.get("p1_theta_y_py", 0.0) for s in stages]
    cs_ty = [s.get("p1_theta_y_cs", 0.0) for s in stages]

    # Transverse tilt theta_x
    ax1.plot(deltas, py_tx, "o-", color="#1f77b4", label=r"Python $	heta_x$", lw=2, ms=6)
    if any(cs_tx):
        ax1.plot(deltas, cs_tx, "s--", color="#d62728", label=r"C# $	heta_x$", lw=1.5, ms=6, alpha=0.8)
    ax1.set_title(r"Transverse Tilt $	heta_x = \Delta U_z / B_{pier}$ (mrad)")
    ax1.set_xlabel("Upstream Scour Depth Ratio")
    ax1.set_ylabel(r"Rotation $	heta_x$ (mrad)")
    ax1.set_xticks(deltas)
    ax1.set_xticklabels(pct_labels)
    ax1.legend()
    ax1.grid(True)

    # Longitudinal tilt theta_y
    ax2.plot(deltas, py_ty, "o-", color="#2ca02c", label=r"Python $	heta_y$", lw=2, ms=6)
    if any(cs_ty):
        ax2.plot(deltas, cs_ty, "s--", color="#ff7f0e", label=r"C# $	heta_y$", lw=1.5, ms=6, alpha=0.8)
    ax2.set_title(r"Longitudinal Tilt $	heta_y = \Delta U_x / H_{pier}$ (mrad)")
    ax2.set_xlabel("Upstream Scour Depth Ratio")
    ax2.set_ylabel(r"Tilt $	heta_y$ (mrad)")
    ax2.set_xticks(deltas)
    ax2.set_xticklabels(pct_labels)
    ax2.legend()
    ax2.grid(True)

    fig.suptitle(title, fontsize=13, fontweight="bold")
    plt.show()


---
# 1. Benchmark 1 (`my_model/benchmark_1`)

**Workflow**: 
1. **Self-weight (`Vert`)**: Nonlinear vertical load analysis.
2. **Scour Phase 1 (`scour_1`)**: Removal of upstream foundation interface stiffness.
3. **Live Load (`LiveLoad_1`)**: Monotonic pushover / live-load sequence.


In [ ]:
# Load Benchmark 1 Model and C# Reference Results
b1_hrx = Path("my_model/benchmark_1/benchmark_virgin.hrx")
b1_results = Path("my_model/benchmark_1/benchmark.Results")

print(f"Loading Benchmark 1: {b1_hrx}...")
b1_model = load_model(b1_hrx)
b1_prep = ModelManager.prepare_model(b1_model)
print(f"Prepared: {b1_prep.gdl} DOFs, {b1_prep.interfaces} interfaces.")

# Execute Sequential Analysis Session in Python
b1_session = AnalysisSession(b1_model)
b1_executions = []

# Step 1: Vert
print("Running Vert...")
b1_executions.append(b1_session.run("Vert"))

# Step 2: Foundation soil / scour mutation
b1_soil_keys = [102, 104, 106, 108]
b1_session.change_interface_materials(b1_soil_keys, SOIL_MATERIAL_KEY)
print("Running scour_1...")
b1_executions.append(b1_session.run("scour_1"))

# Step 3: Live Load
print("Running LiveLoad_1...")
b1_executions.append(b1_session.run("LiveLoad_1"))

# Build Synchronized Python & C# Step History
b1_aligned = build_aligned_step_history(b1_model, b1_executions, b1_results, preferred_mp_key=14)
print(f"Synchronized steps: {len(b1_aligned['steps'])} rows aligned.")


In [ ]:
# Plot: Benchmark 1 Displacement vs Load
plot_displacement_vs_load(
    b1_aligned,
    title="Benchmark 1: Capacity & Reaction Curves (Python vs C#)",
)


---
# 2. Benchmark 2 (`my_model/benchmark_2`)

Multi-span masonry bridge with foundation Soil boundary springs and progressive scour stages.


In [ ]:
# Load Benchmark 2 Model and C# Reference Results
b2_hrx = Path("my_model/benchmark_2/benchmark_virgin.hrx")
b2_results = Path("my_model/benchmark_2/benchmark.Results")

print(f"Loading Benchmark 2: {b2_hrx}...")
b2_model = load_model(b2_hrx)
b2_prep = ModelManager.prepare_model(b2_model)
print(f"Prepared: {b2_prep.gdl} DOFs, {b2_prep.interfaces} interfaces.")

# Resolve foundation interfaces
b2_foundation_keys = [
    k for k, intf in b2_model.collections.interfaces.items()
    if intf.parent_type_element1 == "Restraint" or intf.parent_type_element2 == "Restraint"
]

b2_session = AnalysisSession(b2_model)
b2_session.change_interface_materials(b2_foundation_keys, SOIL_MATERIAL_KEY)

b2_executions = []
print("Running Vert...")
b2_executions.append(b2_session.run("Vert"))

b2_aligned = build_aligned_step_history(b2_model, b2_executions, b2_results, preferred_mp_key=8)

plot_displacement_vs_load(
    b2_aligned,
    title="Benchmark 2: Vertical Equilibrium Curve (Python vs C#)",
)


---
# 3. Full 3D Random Bridge Models (`random_001` through `random_006`)

Each bridge variant contains **6,500+ interfaces** and **17,000+ generalized degrees of freedom**.
We execute the complete 5-step `Vert` baseline followed by progressive scour stages (`scour_1`, `scour_2`), tracking pier displacements, differential settlement, and pier tilt.


In [ ]:
# ==============================================================================
# Select and Execute a Random Model
# ==============================================================================
SELECTED_JOB = "random_005"  # Options: "random_001", "random_002", "random_003", "random_004", "random_005", "random_006"

job_dir = Path("temp-six-jobs") / SELECTED_JOB
if SELECTED_JOB == "random_006":
    rand_hrx = job_dir / "random_005_copy_1.hrx"
    if not rand_hrx.exists():
        rand_hrx = job_dir / "random_005.hrx"
    rand_results = job_dir / "random_005_copy_1.Results"
else:
    rand_hrx = job_dir / f"{SELECTED_JOB}.hrx"
    rand_results = job_dir / f"{SELECTED_JOB}_copy_1.Results"

print(f"[{SELECTED_JOB}] Loading model from {rand_hrx}...")
rand_model = load_model(rand_hrx)
rand_prep = ModelManager.prepare_model(rand_model)
print(f"[{SELECTED_JOB}] Prepared: {rand_prep.gdl} DOFs, {rand_prep.interfaces} interfaces.")

# Scour interface key resolution
soil_keys = pier_foundation_interface_keys(rand_model)
scour_1_keys = upstream_interface_keys(rand_model, 0.2)
scour_2_keys = upstream_interface_keys(rand_model, 0.4)

print(f"[{SELECTED_JOB}] Soil keys: {len(soil_keys)}, Scour keys: 20%={len(scour_1_keys)}, 40%={len(scour_2_keys)}")

rand_session = AnalysisSession(rand_model)
if soil_keys:
    rand_session.change_interface_materials(soil_keys, SOIL_MATERIAL_KEY, preserve_committed_state=False)

rand_executions = []
print(f"[{SELECTED_JOB}] 1. Running Vert...")
rand_executions.append(rand_session.run("Vert"))

if scour_1_keys:
    print(f"[{SELECTED_JOB}] 2. Running scour_1 (20%)...")
    rand_session.change_interface_materials(scour_1_keys, SOIL_REMOVED_MATERIAL_KEY)
    rand_executions.append(rand_session.run("scour_1"))

if scour_2_keys:
    print(f"[{SELECTED_JOB}] 3. Running scour_2 (40%)...")
    rand_session.change_interface_materials(scour_2_keys, SOIL_REMOVED_MATERIAL_KEY)
    rand_executions.append(rand_session.run("scour_2"))

# 1. Build Aligned Step History across All Sequential Analyses
rand_aligned = build_aligned_step_history(rand_model, rand_executions, rand_results, preferred_mp_key=8)

# 2. Extract Final Converged Equilibrium States for Scour Progression
scour_stages = [(0.0, "Vert"), (0.2, "scour_1"), (0.4, "scour_2")]
rand_scour = extract_scour_progression(rand_model, rand_executions, rand_results, scour_stages)

print(f"[{SELECTED_JOB}] Extraction complete. {len(rand_aligned['steps'])} aligned steps, {len(rand_scour['stages'])} scour stages.")


In [ ]:
# Plot 1: Displacement vs Load for Selected Random Model
plot_displacement_vs_load(
    rand_aligned,
    title=f"{SELECTED_JOB}: Displacement vs Reaction Curves (Python vs C#)",
)


In [ ]:
# Plot 2: Scour Depth Ratio vs Pier Settlement & Transverse Drift
plot_scour_vs_displacement(
    rand_scour,
    title=f"{SELECTED_JOB}: Scour Depth Ratio vs Pier 1 Displacements",
)


In [ ]:
# Plot 3: 3-Axis Pier Top Kinematics (Ux, Uy, Uz) across Analysis Steps
plot_pier_top_kinematics(
    rand_aligned,
    title=f"{SELECTED_JOB}: 3-Axis Pier Top Kinematics under Load & Scour",
    pier_mp=8,
)

# Plot 4: Pier Tilt and Rotation Angles (theta_x, theta_y in mrad)
plot_pier_rotation(
    rand_scour,
    title=f"{SELECTED_JOB}: Pier Tilt (Rotation Angles in mrad) vs Scour Depth",
)


---
# 4. Multi-Model Batch Parity Benchmark (All 6 Models)

Comparative summary of **maximum displacement node deflections**, **peak discrepancies**, and **reaction totals** across all 6 random models.


In [ ]:
# Run / Compare all 6 models for Vert baseline
batch_results = []

for job_idx in range(1, 7):
    jname = f"random_{job_idx:03d}"
    jdir = Path("temp-six-jobs") / jname
    
    if job_idx == 6:
        hrx_file = jdir / "random_005_copy_1.hrx"
        if not hrx_file.exists():
            hrx_file = jdir / "random_005.hrx"
        res_file = jdir / "random_005_copy_1.Results"
    else:
        hrx_file = jdir / f"{jname}.hrx"
        res_file = jdir / f"{jname}_copy_1.Results"

    print(f"Evaluating {jname}...", end=" ", flush=True)
    m = load_model(hrx_file)
    ModelManager.prepare_model(m)
    sess = AnalysisSession(m)
    skeys = pier_foundation_interface_keys(m)
    if skeys:
        sess.change_interface_materials(skeys, SOIL_MATERIAL_KEY, preserve_committed_state=False)
    
    exec_vert = sess.run("Vert")
    py_u = exec_vert.committed_steps[-1].u
    py_r3 = exec_vert.committed_steps[-1].reaction_z

    cs_u = np.zeros_like(py_u)
    cs_r3 = 0.0
    if res_file.exists():
        with sqlite3.connect(res_file) as db:
            v_rows = db.execute("SELECT Dof, U FROM DynamicVectorsState WHERE AnalysisKey=1 AND Combination=1 ORDER BY Dof").fetchall()
            if v_rows:
                cs_u = np.array([r[1] for r in v_rows], dtype=float)
            r_rows = db.execute("SELECT R3 FROM ReactionSumStates WHERE AnalysisKey=1 AND Combination=1 ORDER BY Step DESC LIMIT 1").fetchone()
            if r_rows:
                cs_r3 = r_rows[0]

    diff = py_u - cs_u
    max_disp_idx = int(np.argmax(np.abs(cs_u)))
    max_disc_idx = int(np.argmax(np.abs(diff)))

    batch_results.append({
        "model": jname,
        "peak_csharp_mm": float(cs_u[max_disp_idx]) * 1000.0,
        "peak_python_mm": float(py_u[max_disp_idx]) * 1000.0,
        "peak_diff_mm": float(py_u[max_disp_idx] - cs_u[max_disp_idx]) * 1000.0,
        "peak_rel_pct": abs(py_u[max_disp_idx] - cs_u[max_disp_idx]) / abs(cs_u[max_disp_idx]) * 100.0 if cs_u[max_disp_idx] else 0.0,
        "max_discrepancy_mm": float(np.max(np.abs(diff))) * 1000.0,
        "reaction_diff_pct": abs(py_r3 - cs_r3) / abs(cs_r3) * 100.0 if cs_r3 else 0.0,
    })
    print(f"Peak Diff = {batch_results[-1]['peak_diff_mm']:+.3f} mm ({batch_results[-1]['peak_rel_pct']:.2f}%), Max Disc = {batch_results[-1]['max_discrepancy_mm']:.3f} mm")

print("\nBatch evaluation complete.")


In [ ]:
# Plot Multi-Model Parity Summary
models = [r["model"] for r in batch_results]
cs_peaks = [abs(r["peak_csharp_mm"]) for r in batch_results]
py_peaks = [abs(r["peak_python_mm"]) for r in batch_results]
diffs_mm = [r["max_discrepancy_mm"] for r in batch_results]
rel_errs = [r["peak_rel_pct"] for r in batch_results]

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 4.5))

# 1. Peak Deflection Comparison
x = np.arange(len(models))
width = 0.35
ax1.bar(x - width/2, cs_peaks, width, label="C# HiStrA", color="#d62728", alpha=0.85)
ax1.bar(x + width/2, py_peaks, width, label="Python Solver", color="#1f77b4", alpha=0.85)
ax1.set_title("Peak Vertical Deflection |Uz| (mm)")
ax1.set_ylabel("Deflection (mm)")
ax1.set_xticks(x)
ax1.set_xticklabels(models, rotation=30)
ax1.legend()
ax1.grid(True)

# 2. Maximum Discrepancy (mm)
ax2.bar(models, diffs_mm, color="#ff7f0e", alpha=0.85)
ax2.set_title("Max Displacement Discrepancy Across DOFs")
ax2.set_ylabel("Max |du| (mm)")
ax2.set_xticks(x)
ax2.set_xticklabels(models, rotation=30)
ax2.grid(True)

# 3. Relative Difference at Peak Node (%)
ax3.bar(models, rel_errs, color="#2ca02c", alpha=0.85)
ax3.set_title("Relative Error at Peak Node (%)")
ax3.set_ylabel("Relative Difference (%)")
ax3.set_xticks(x)
ax3.set_xticklabels(models, rotation=30)
ax3.grid(True)

fig.suptitle("6 Random Models: Python vs C# Parity Summary", fontsize=14, fontweight="bold")
plt.show()
